# Lesson 10 | Why do spikes need a queue?

A shared engine may already be busy when new spikes arrive. Today asks:
> **When events temporarily arrive faster than they can be consumed, how do we preserve order without silently losing them?**

Primary concept: a **bounded First-In, First-Out queue (FIFO)**.


## 1. Concept ledger

**Known:** a spike is a discrete event; a shared engine may be busy.

**New:** event queue, FIFO, full/empty, and backpressure.

**Preview:** the next lesson explains how a source spike finds downstream synapses.


## 2. What is an event?

Here a spike becomes a tiny **event record**. The minimal record contains only `source_id`: which neuron spiked. Later systems may add time or metadata, but this lesson does not need them.


## 3. What does FIFO guarantee?

**First-In, First-Out (FIFO):** the earliest arriving event leaves first.

If events enter as `2, 5, 7`, normal pop order must also be `2, 5, 7`. FIFO is about buffering and order, not the biological meaning of a spike.


## 4. Why does a bounded queue need backpressure?

A real hardware queue has finite capacity. When full, the producer must not pretend the next event was accepted.

**Backpressure** is a control condition telling the producer “not ready yet; hold/wait.” In the Python model, `accepted=False` represents that condition.

```mermaid
flowchart LR
 P["spike producer"] -->|push event| Q["bounded FIFO"]
 Q -->|pop event| C["consumer"]
 Q -. "full / not ready" .-> P
```

In this teaching API, `accepted=False` means **this push did not happen**. The caller still owns the event and must not forget it; after space opens, the same event should be retried. It is not permission to drop the spike.


## 5. Run: a queue with capacity 3

Predict what happens to the fourth event, `9`, then run.


In [ ]:
from collections import deque

capacity = 3
q = deque()

def show(action):
    print(f'{action:18s} queue={list(q)} full={len(q) == capacity} empty={len(q) == 0}')

for event in [2, 5, 7, 9]:
    if len(q) < capacity:
        q.append(event)
        show(f'accepted spike {event}')
    else:
        show(f'blocked spike {event}')

while q:
    event = q.popleft()
    show(f'consumed spike {event}')


## 6. Observe

The first three events are accepted. Once full, the fourth event does not overwrite older events. Pops still preserve arrival order.

**Important:** “blocked” does not mean “safe to drop.” A formal interface must make the producer hold the event until ready, or buffer it farther upstream.

The public check therefore also verifies that an event blocked by a full queue can be retried unchanged after a pop creates space.


## 7. Try It: slow consumer

Change capacity to 2. Predict which push blocks first. Then try “push two → pop one → push one” and observe backpressure being released.


## 8. Homework

Complete `fifo_push(...)` and `fifo_pop(...)` in `exercises/lesson10_fifo.py`. Public checks cover FIFO ordering, empty pop, a full queue leaving both the queue and caller-owned event intact, and successful retry after space opens.

```bash
uv run pytest exercises/checks/check_lesson10.py -q
```


## 9. AI Task

Ask AI for the shortest capacity-2 operation sequence that visits empty, non-empty, full, and backpressured states. Require expected queue contents before code.


## 10. Human Check

Without AI, explain why FIFO preserves order, what full/empty mean, why backpressure is safer than overwriting an old spike, and why a queue transports events rather than changing neuron semantics.


## 11. Engineering Handoff

This lesson aligns conceptually with `MOD-005 spike_fifo`, `IF-SPIKE-QUEUE`, T-007 FIFO ordering, and T-008 FIFO backpressure. It is not yet the formal RTL FIFO.


## 12. Project Trace

- Lesson: `LSN-010`
- Mapping: `RMD-007A / RMD-009` teaching precursor
- Module context: `MOD-005`
- Test context: `T-007 / T-008`


## 13. Exit Ticket

You can explain bounded-FIFO ordering and backpressure, and recognize why a full queue must explicitly delay or reject a new event rather than silently overwrite data.
